In [1]:
# Clone the repository
!git clone https://github.com/METR/eval-analysis-public.git
%cd eval-analysis-public

# Install the horizon package in editable mode
!pip install -e .

fatal: destination path 'eval-analysis-public' already exists and is not an empty directory.
/content/eval-analysis-public
Obtaining file:///content/eval-analysis-public
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for horizon (pyproject.toml) ... done
  Created wheel for horizon: filename=horizon-0.1.0-0.editable-py3-none-any.whl size=3279 sha256=db2f60017a01f780a62ac4c6ba352fca5f1c8be938801057152976718d9248f0
  Stored in directory: /tmp/pip-ephem-wheel-cache-u24s25gc/wheels/8c/4a/8f/beba6aa9b8b9a057d3c6e7fdc67f23bdbcfac7f32b3480870e
Successfully built horizon
  Attempting uninstall: horizon
    Found existing installation: horizon 0.1.0
    Uninstalling horizon-0.1.0:
      Successfully uninstalled horizon-0.1.0


In [2]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# Load raw data (use 1-1 for latest)
runs = pd.read_json("reports/time-horizon-1-1/data/raw/runs.jsonl", lines=True)

# Load release dates for trend fitting
import yaml
with open("data/external/release_dates.yaml") as f:
    release_dates = yaml.safe_load(f)

# Logistic function: P(success) as function of log2(human_minutes)
def logistic(x, x0, k):
    return 1 / (1 + np.exp(-k * (x - x0)))

# For each model, fit curve and extract time horizon at p=0.99
results = []
for model, group in runs.groupby("alias"):
    x = np.log2(group["human_minutes"].values)
    y = group["score_binarized"].values
    try:
        popt, _ = curve_fit(logistic, x, y, p0=[5, -1], maxfev=5000)
        x0, k = popt
        # Horizon = log2(minutes) where P(success) = threshold
        for threshold in [0.50, 0.80, 0.99]:
            horizon_log2 = x0 - np.log(1/threshold - 1) / k
            horizon_minutes = 2**horizon_log2
            results.append({
                "model": model,
                "threshold": threshold,
                "horizon_minutes": horizon_minutes
            })
    except:
        pass

df = pd.DataFrame(results)
print(df[df["threshold"] == 0.99].sort_values("horizon_minutes"))

                                model  threshold  horizon_minutes
53                              human       0.99         0.004560
2             Claude 3 Opus (Inspect)       0.99         0.004702
29               GPT-4 1106 (Inspect)       0.99         0.006456
8   Claude 3.5 Sonnet (Old) (Inspect)       0.99         0.007856
5   Claude 3.5 Sonnet (New) (Inspect)       0.99         0.009519
26                         GPT-4 0314       0.99         0.010726
35                   GPT-4o (Inspect)       0.99         0.013151
32              GPT-4 Turbo (Inspect)       0.99         0.019185
59                         o1-preview       0.99         0.087926
56                       o1 (Inspect)       0.99         0.302106
11        Claude 3.7 Sonnet (Inspect)       0.99         0.602274
38                    GPT-5 (Inspect)       0.99         0.874052
23          Claude Opus 4.6 (Inspect)       0.99         1.155192
14            Claude 4 Opus (Inspect)       0.99         1.203961
20        

In [4]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.stats import linregress
from datetime import datetime
import yaml

# Load data
runs = pd.read_json("reports/time-horizon-1-1/data/raw/runs.jsonl", lines=True)
with open("data/external/release_dates.yaml") as f:
    release_dates = yaml.safe_load(f)["date"]

# Logistic function
def logistic(x, x0, k):
    return 1 / (1 + np.exp(-k * (x - x0)))

# Fit p99 horizon for each model
results = []
for model, group in runs.groupby("alias"):
    x = np.log2(group["human_minutes"].values)
    y = group["score_binarized"].values
    if len(group) < 10 or y.mean() == 0 or y.mean() == 1:
        continue
    try:
        popt, _ = curve_fit(logistic, x, y, p0=[5, -1], maxfev=5000,
                    sigma=1/group["invsqrt_task_weight"].values,
                    absolute_sigma=False)
        x0, k = popt
        if k >= 0:
            continue
        horizon_log2 = x0 - np.log(1/0.99 - 1) / k
        results.append({
            "model": model,
            "horizon_minutes": 2**horizon_log2,
            "release_date": release_dates.get(model)
        })
    except:
        pass

p99 = pd.DataFrame(results).dropna(subset=["release_date"])
p99["release_date"] = pd.to_datetime(p99["release_date"])

# Fit exponential trend
epoch = datetime(2019, 1, 1)
p99["days"] = (p99["release_date"] - epoch).dt.days
p99["log2_horizon"] = np.log2(p99["horizon_minutes"])
p99 = p99.sort_values("horizon_minutes", ascending=False).drop_duplicates(subset="release_date", keep="first")

slope, intercept, r_value, _, _ = linregress(p99["days"], p99["log2_horizon"])

print(f"Doubling time: {1/slope:.0f} days ({1/slope/30.44:.1f} months)")
print(f"R² = {r_value**2:.4f}")
print(f"Current best p99: {p99['horizon_minutes'].max():.2f} minutes\n")

for year, month in [(2026,6), (2027,1), (2027,6), (2028,1), (2028,6), (2029,1), (2030,1), (2030,12)]:
    d = (datetime(year, month, 1) - epoch).days
    mins = 2**(slope * d + intercept)
    label = f"{mins/60:.1f} hours" if mins >= 60 else f"{mins:.1f} minutes"
    print(f"{datetime(year,month,1).strftime('%b %Y')}: p99 ≈ {label}")

Doubling time: 93 days (3.1 months)
R² = 0.8125
Current best p99: 2.03 minutes

Jun 2026: p99 ≈ 6.1 minutes
Jan 2027: p99 ≈ 29.8 minutes
Jun 2027: p99 ≈ 1.5 hours
Jan 2028: p99 ≈ 7.5 hours
Jun 2028: p99 ≈ 23.0 hours
Jan 2029: p99 ≈ 112.8 hours
Jan 2030: p99 ≈ 1693.5 hours
Dec 2030: p99 ≈ 20203.6 hours


In [5]:
!cat LICENSE

cat: LICENSE: No such file or directory
